In [ ]:
# 1. Удаляем все остатки старых библиотек
!pip uninstall -y transformers tokenizers datasets accelerate

# 2. Устанавливаем чистые и актуальные версии
!pip install "transformers>=4.48.0" "tokenizers>=0.21.0" "datasets>=3.0.0" accelerate

# 3. ВАЖНО: После выполнения этой ячейки нажмите в меню:
# Runtime -> Restart session (или Среда выполнения -> Перезапустить сессию)

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 11.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import math
import torch
import random
import numpy as np
import gc
from datasets import load_dataset
from transformers import (
    BertConfig,
    BertForMaskedLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments
)
model_id = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

In [ ]:
BASE_DIR = "/content/drive/MyDrive/AncientRusProject"
DATA_FILE = f"{BASE_DIR}/ancient_rus_ready_for_bert.txt"
TOKENIZER_DIR = f"{BASE_DIR}/ancient_rus_tokenizer"
MODEL_DIR = f"{BASE_DIR}/mini_roformer_ancient_rus"
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
print("⏳ Загрузка BPE токенизатора...")
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_DIR,
    use_fast=True,           # ModernBERT любит быстрые токенизаторы
    trust_remote_code=True   # На всякий случай для новых архитектур
)

⏳ Загрузка BPE токенизатора...


In [ ]:
special_tokens_dict = {
    'additional_special_tokens': [
        "[CTX_CHURCH]", "[CTX_DAILY]", "[CTX_LEGAL]",
        "[CTX_LIT]", "[CTX_EPIC]", "[CTX_SCIENCE]", "[UNK]"
    ]
}

In [ ]:
tokenizer.add_special_tokens(special_tokens_dict)

1

In [ ]:
dataset = load_dataset("text", data_files={"train": DATA_FILE})
split_dataset = dataset["train"].train_test_split(test_size=0.05, seed=42)

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=False)

In [ ]:
tokenized_datasets = split_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

Map:   0%|          | 0/465075 [00:00<?, ? examples/s]

Map:   0%|          | 0/24478 [00:00<?, ? examples/s]

In [ ]:
def group_texts(examples):
    block_size = 256
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = (len(concatenated_examples[list(examples.keys())[0]]) // block_size) * block_size
    return {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }

In [ ]:
lm_datasets = tokenized_datasets.map(group_texts, batched=True)

Map:   0%|          | 0/465075 [00:00<?, ? examples/s]

Map:   0%|          | 0/24478 [00:00<?, ? examples/s]

In [ ]:
class PhysicalDegradationCollator:
    """
    Симулирует реальные повреждения исторических документов:
    1. Отломанные края (Edge Masking)
    2. Вытертые дыры (Span Masking)
    3. Стертые части слов (Random Subword Masking)
    """
    def __init__(self, tokenizer, mlm_prob=0.15, max_span=3, edge_prob=0.1):
        self.tokenizer = tokenizer
        self.mlm_prob = mlm_prob
        self.max_span = max_span
        self.edge_prob = edge_prob # Вероятность, что у предложения оторван край

    def __call__(self, features):
        input_ids = torch.tensor([f["input_ids"] for f in features], dtype=torch.long)
        attention_mask = torch.tensor([f["attention_mask"] for f in features], dtype=torch.long)
        labels = input_ids.clone()

        batch_size, seq_len = input_ids.shape
        probability_matrix = torch.full(labels.shape, self.mlm_prob)

        # Защита спецтокенов
        special_tokens_mask = [
            self.tokenizer.get_special_tokens_mask(val, already_has_special_tokens=True) for val in labels.tolist()
        ]
        special_tokens_mask = torch.tensor(special_tokens_mask, dtype=torch.bool)
        probability_matrix.masked_fill_(special_tokens_mask, value=0.0)

        # Базовая случайная маска
        masked_indices = torch.bernoulli(probability_matrix).bool()
        final_mask = masked_indices.clone()



        for i in range(batch_size):
            # 1. Edge Masking (Отломанный край)
            if random.random() < self.edge_prob:
                edge_len = random.randint(2, 5)
                is_start = random.choice([True, False])

                # Ищем границы, игнорируя <s> и </s> и теги контекста
                valid_indices = (~special_tokens_mask[i]).nonzero(as_tuple=True)[0]
                if len(valid_indices) > edge_len:
                    if is_start:
                        start_idx = valid_indices[0]
                        final_mask[i, start_idx : start_idx + edge_len] = True
                    else:
                        end_idx = valid_indices[-1]
                        final_mask[i, end_idx - edge_len + 1 : end_idx + 1] = True

            # 2. Span Masking (Вытертые дыры)
            for j in range(seq_len):
                if masked_indices[i, j]:
                    span_len = random.randint(1, self.max_span)
                    end_idx = min(j + span_len, seq_len)
                    if not special_tokens_mask[i, j:end_idx].any():
                        final_mask[i, j:end_idx] = True

        labels[~final_mask] = -100

        # Стандартные 80% [MASK], 10% рандом, 10% оригинал
        indices_replaced = torch.bernoulli(torch.full(labels.shape, 0.8)).bool() & final_mask
        input_ids[indices_replaced] = self.tokenizer.mask_token_id

        indices_random = torch.bernoulli(torch.full(labels.shape, 0.5)).bool() & final_mask & ~indices_replaced
        random_words = torch.randint(len(self.tokenizer), labels.shape, dtype=torch.long)
        input_ids[indices_random] = random_words[indices_random]

        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

In [ ]:
data_collator = PhysicalDegradationCollator(tokenizer=tokenizer, mlm_prob=0.12, max_span=3, edge_prob=0.15)

In [ ]:
config = BertConfig(
    vocab_size=len(tokenizer),
    hidden_size=512,
    num_hidden_layers=8, # ModernBERT эффективен даже с меньшим числом слоев
    num_attention_heads=8,
    intermediate_size=2048,
    max_position_embeddings=1024, # ModernBERT отлично держит длинный контекст
    pad_token_id=tokenizer.pad_token_id,
)

In [ ]:
from transformers import AutoConfig, AutoModelForMaskedLM

model_id = "answerdotai/ModernBERT-base"
config = AutoConfig.from_pretrained(model_id)

# Вариант 1: Уменьшаем количество голов до 8 (512 / 8 = 64 — идеально)
config.num_attention_heads = 8
config.hidden_size = 512
config.num_hidden_layers = 8
config.intermediate_size = 2048 # В ModernBERT обычно в 4 раза больше hidden_size

# Инициализируем модель
model = AutoModelForMaskedLM.from_config(config)

# Подгоняем эмбеддинги
model.resize_token_embeddings(len(tokenizer))

print(f"✅ Модель успешно создана!")
print(f"🚀 Параметры ModernBERT: {model.num_parameters():,}")
print(f"📐 Размерность одной головы: {config.hidden_size // config.num_attention_heads}")

✅ Модель успешно создана!
🚀 Параметры ModernBERT: 49,217,331
📐 Размерность одной головы: 64


In [ ]:
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return torch.topk(logits, k=5, dim=-1).indices

In [ ]:
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    mask = labels != -100
    labels = labels[mask]
    preds = preds[mask]

    return {
        "top1_accuracy": np.mean(preds[:, 0] == labels),
        "top3_accuracy": np.mean(np.any(preds[:, :3] == labels[:, None], axis=1)),
        "top5_accuracy": np.mean(np.any(preds[:, :5] == labels[:, None], axis=1)),
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    #overwrite_output_dir=True,          # Должно работать, если библиотека обновилась корректно
    num_train_epochs=15,
    per_device_train_batch_size=64,
    gradient_accumulation_steps=2,
    eval_strategy="steps",              # Замените evaluation_strategy на eval_strategy
    eval_steps=400,
    save_steps=400,
    save_total_limit=2,
    logging_steps=100,
    # prediction_loss_only=False,       # Можно убрать, это значение по умолчанию
    learning_rate=5e-4,
    lr_scheduler_type="cosine",
    warmup_steps=1000,
    weight_decay=0.01,
    fp16=True,                          # Если у вас T4 в Colab. Если A100/H100 — ставьте bf16=True
    dataloader_num_workers=2,
    report_to="none",
    load_best_model_at_end=True,
    # Добавьте это для ModernBERT:
    bf16=False,                         # ModernBERT любит bf16, но на T4 (Colab) оставляем fp16=True
    optim="adamw_torch_fused",          # Ускоряет обучение на новых версиях
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics
)

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None, 'pad_token_id': 0}.


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

In [ ]:
print(f"\n📊 ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ:")
eval_results = trainer.evaluate()
print(f"Loss: {eval_results['eval_loss']:.4f}")
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")
print(f"Top-5 Точность: {eval_results.get('eval_top5_accuracy', 0):.2%}")

In [ ]:
from transformers import pipeline

In [ ]:
roformer_pipe = pipeline(
    "fill-mask",
    model=MODEL_DIR,
    tokenizer=MODEL_DIR,
    device=0 # Если GPU доступен, иначе ставь -1
)

# Хардкорные тесты (Обрати внимание: для RoFormer мы используем <mask >)
test_cases = [
    # 1. Классика: проверка падежей и логики (Летописи)
    {
        "desc": "📚 Летописи (на какую землю?)",
        "text": "[CTX_LIT] И пошелъ князь игорь на <mask> землю со своею дружиною.",
        "expected": "рускую / свою"
    },

    # 2. Судебник: проверка знания конкретных законов
    {
        "desc": "⚖️ Русская Правда (кого убили?)",
        "text": "[CTX_LEGAL] Аже кто оубиеть <mask> , то платити виру 40 гривенъ.",
        "expected": "мужь"
    },

    # 3. Бытовой: проверка понимания долгов
    {
        "desc": "🏡 Грамоты (про что пишут?)",
        "text": "[CTX_DAILY] поклоне ѿ ꙩндреꙗ · к ѥва · и к микифору про <mask> ѡкупи ꙩсподине",
        "expected": "серебро"
    },

    # 4. ТЕСТ НА ОТОРВАННЫЙ КРАЙ (Edge Masking)
    # У предложения нет начала, но RoPE должен понять, что к Василию шлют поклон
    {
        "desc": "🧨 ТЕСТ ROPE: Оторванное начало",
        "text": "[CTX_DAILY] <mask> <mask> ко василью . а серебро ми отдай.",
        "expected": "поклонъ ѿ"
    },

    # 5. ТЕСТ НА ОТОРВАННЫЙ КОНЕЦ (Edge Masking)
    {
        "desc": "🧨 ТЕСТ ROPE: Оторванный конец",
        "text": "[CTX_EPIC] Выезжал добрый <mask> из <mask> на <mask> <mask>",
        "expected": "молодец из города на добром коне"
    },

    # 6. ТЕСТ НА [UNK] (Реальная деградация из твоего датасета)
    # Проверяем, не сойдет ли модель с ума от тега [UNK]
    {
        "desc": "🧩 ТЕСТ [UNK]: Работа с нечитаемым текстом",
        "text": "[CTX_DAILY] [UNK] бь ѿ но [UNK] тию и св <mask> коуно",
        "expected": "Модель должна предложить варианты, игнорируя дыры [UNK]"
    },

    # 7. Церковный: проверка множественного числа
    {
        "desc": "⛪️ Церковный (кому сказал?)",
        "text": "[CTX_CHURCH] И рече господь къ <mask> своимъ, глаголя...",
        "expected": "ученикомъ / людемъ"
    }
]

print("\n" + "=" * 60)
print("🚀 СТАРТ ТЕСТИРОВАНИЯ ROFORMER (RoPE + DEGRADATION)")
print("=" * 60)

for idx, case in enumerate(test_cases, 1):
    print(f"\n[{idx}/7] {case['desc']}")
    print(f"📝 Текст: {case['text']}")
    print(f"🎯 Ожидалось (смысл): {case['expected']}")

    # Если масок несколько, пайплайн вернет список списков
    mask_count = case['text'].count("<mask>")
    results = roformer_pipe(case['text'], top_k=3)

    # Нормализуем вывод (если маска одна, оборачиваем в список для единообразия)
    if mask_count == 1:
        results = [results]

    for i, mask_res in enumerate(results):
        print(f"  ➡️ Маска {i+1}: ", end="")
        preds = []
        for res in mask_res:
            # Очищаем от байтового пробела RoBERTa/RoFormer
            clean_word = res['token_str'].replace("Ġ", "").strip()
            score = res['score'] * 100
            preds.append(f"'{clean_word}' ({score:.1f}%)")
        print(" | ".join(preds))

In [ ]:
# Запуск твоего тренировочного скрипта с параметрами
!python train_ner.py --data_path data.json --epochs 3 --batch_size 16 --output_dir ./ner_model

Found tags: {'B-ANIMAL': 0, 'I-ANIMAL': 1, 'O': 2}
Loading weights: 100% 197/197 [00:00<00:00, 1321.44it/s, Materializing param=bert.encoder.layer.11.output.dense.weight]
BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

In [ ]:
import shutil
from google.colab import files

# 1. Запаковываем папку с моделью в zip-архив
shutil.make_archive('my_ner_model', 'zip', './ner_model')

# 2. Скачиваем полученный архив на компьютер
files.download('my_ner_model.zip')

KeyboardInterrupt: 

In [ ]:
# Запуск твоего тренировочного скрипта с параметрами
!python train_ner.py --data_path data.json --epochs 3 --batch_size 16 --output_dir ./ner_model

Found tags: {'B-ANIMAL': 0, 'I-ANIMAL': 1, 'O': 2}
Loading weights: 100% 197/197 [00:00<00:00, 421.43it/s, Materializing param=bert.encoder.layer.11.output.dense.weight]
BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 


In [ ]:
!python train_image_classifier.py --data_dir "/content/train_data/animals/animals" --epochs 5 --batch_size 32 --output_model "train_image_classifier.pth"

Using NVIDIA GPU (CUDA)
Loading data...
Classes found (90): ['antelope', 'badger', 'bat', 'bear', 'bee', 'beetle', 'bison', 'boar', 'butterfly', 'cat', 'caterpillar', 'chimpanzee', 'cockroach', 'cow', 'coyote', 'crab', 'crow', 'deer', 'dog', 'dolphin', 'donkey', 'dragonfly', 'duck', 'eagle', 'elephant', 'flamingo', 'fly', 'fox', 'goat', 'goldfish', 'goose', 'gorilla', 'grasshopper', 'hamster', 'hare', 'hedgehog', 'hippopotamus', 'hornbill', 'horse', 'hummingbird', 'hyena', 'jellyfish', 'kangaroo', 'koala', 'ladybugs', 'leopard', 'lion', 'lizard', 'lobster', 'mosquito', 'moth', 'mouse', 'octopus', 'okapi', 'orangutan', 'otter', 'owl', 'ox', 'oyster', 'panda', 'parrot', 'pelecaniformes', 'penguin', 'pig', 'pigeon', 'porcupine', 'possum', 'raccoon', 'rat', 'reindeer', 'rhinoceros', 'sandpiper', 'seahorse', 'seal', 'shark', 'sheep', 'snake', 'sparrow', 'squid', 'squirrel', 'starfish', 'swan', 'tiger', 'turkey', 'turtle', 'whale', 'wolf', 'wombat', 'woodpecker', 'zebra']
Loading pre-trained

In [ ]:
!unzip -q /content/animals.zip -d /content/train_data